# 01 - TF-IDF baseline on cleaned full AllNLI

Notebook nay chay doc lap tren Kaggle. TF-IDF la baseline khong lap theo epoch nen khong co early stopping; notebook van danh gia test 5k va thoi gian scoring tren test 5k.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
GITHUB_REPOSITORY_URL = 'https://github.com/PhDQuang/similarity_search.git'

if not PROJECT_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
%pip install -q -e fix

In [ ]:
from pathlib import Path
import shutil

CLEAN_DATA_DIR = Path('fix/data/processed/allnli_70_15_15_clean/pair-class')
KAGGLE_CLEAN_CANDIDATES = [
    Path('/kaggle/input/allnli-70-15-15-clean/pair-class'),
    Path('/kaggle/input/allnli-70-15-15-clean/allnli_70_15_15_clean/pair-class'),
]

def has_clean_data(path: Path) -> bool:
    return all((path / f'{split}.parquet').exists() for split in ('train', 'val', 'test'))

if not has_clean_data(CLEAN_DATA_DIR):
    source = next((path for path in KAGGLE_CLEAN_CANDIDATES if has_clean_data(path)), None)
    if source is not None:
        CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for item in source.iterdir():
            if item.is_file():
                shutil.copy2(item, CLEAN_DATA_DIR / item.name)
    else:
        !python -m similarity_search_fix.data.prepare_allnli_70_15_15_clean --output-dir {CLEAN_DATA_DIR} --seed 42

assert has_clean_data(CLEAN_DATA_DIR), f'Missing clean data: {CLEAN_DATA_DIR}'
print('Using clean data:', CLEAN_DATA_DIR)

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/fix_outputs/tfidf_clean_full')
MODEL_DIR = Path('/kaggle/working/fix_models/tfidf_clean_full')

!python -m similarity_search_fix.models.train_tfidf \
  --input-dir {CLEAN_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --model-dir {MODEL_DIR} \
  --max-features 50000 \
  --ngram-max 2 \
  --max-retrieval-queries 1000 \
  --test-sample-size 5000 \
  --seed 42

In [ ]:
from pathlib import Path
import json
import shutil

ARTIFACT_DIR = Path('/kaggle/working/artifacts_tfidf_clean_full')
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
shutil.copytree(OUTPUT_DIR, ARTIFACT_DIR / 'outputs')
shutil.copytree(MODEL_DIR, ARTIFACT_DIR / 'model')
zip_path = shutil.make_archive(str(ARTIFACT_DIR), 'zip', ARTIFACT_DIR)
print('Download artifact:', zip_path)

display(json.loads((OUTPUT_DIR / 'metrics.json').read_text()))
display(json.loads((OUTPUT_DIR / 'test5k_performance.json').read_text()))